# GlCM setup and path 

In [1]:
# ============================================================
# CELL 1: GLCM FEATURE EXTRACTION SETUP
# ============================================================

import os
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

from skimage.feature import graycomatrix, graycoprops


# ============================================================
# 1. DATASET PATH
# ============================================================

DATASET_PATH = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET"
)


# ============================================================
# 2. SAVED TUMOR ROI DATASET
# ============================================================

ROI_DATASET_PATH = (
    DATASET_PATH / "Tumor_ROI_Morphology"
)


# ============================================================
# 3. TRAIN AND TEST PATHS
# ============================================================

ROI_TRAIN_PATH = (
    ROI_DATASET_PATH / "train"
)

ROI_TEST_PATH = (
    ROI_DATASET_PATH / "test"
)


# ============================================================
# 4. FEATURE OUTPUT DIRECTORY
# ============================================================

FEATURE_PATH = (
    DATASET_PATH / "Features(1)"
)

FEATURE_PATH.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. GLCM PARAMETERS
# ============================================================

# Distances between pixels
GLCM_DISTANCES = [
    1,
    2,
    3,
    4
]


# Directions / angles
GLCM_ANGLES = [
    0,
    np.pi / 4,
    np.pi / 2,
    3 * np.pi / 4
]


# Number of gray levels
# Reducing from 256 to 32 makes GLCM faster
GLCM_LEVELS = 32


# ============================================================
# 6. GLCM FEATURES
# ============================================================

GLCM_PROPERTIES = [
    "contrast",
    "dissimilarity",
    "homogeneity",
    "energy",
    "correlation",
    "ASM"
]


# ============================================================
# 7. DISPLAY CONFIGURATION
# ============================================================

print("=" * 60)
print("GLCM FEATURE EXTRACTION SETUP")
print("=" * 60)

print("\nROI Dataset:")
print(ROI_DATASET_PATH)

print("\nTrain ROI:")
print(ROI_TRAIN_PATH)

print("\nTest ROI:")
print(ROI_TEST_PATH)

print("\nFeature Output:")
print(FEATURE_PATH)

print("\nGLCM Distances:")
print(GLCM_DISTANCES)

print("\nGLCM Angles:")
print(GLCM_ANGLES)

print("\nGLCM Gray Levels:")
print(GLCM_LEVELS)

print("\nGLCM Properties:")
print(GLCM_PROPERTIES)

GLCM FEATURE EXTRACTION SETUP

ROI Dataset:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Tumor_ROI_Morphology

Train ROI:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Tumor_ROI_Morphology\train

Test ROI:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Tumor_ROI_Morphology\test

Feature Output:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)

GLCM Distances:
[1, 2, 3, 4]

GLCM Angles:
[0, 0.7853981633974483, 1.5707963267948966, 2.356194490192345]

GLCM Gray Levels:
32

GLCM Properties:
['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']


# ROI crop + GLCM feature extraction

In [2]:
# ============================================================
# CELL 2: ROI CROPPING + GLCM FEATURE EXTRACTION FUNCTIONS
# ============================================================

import numpy as np
import cv2

from skimage.feature import graycomatrix, graycoprops


# ============================================================
# 1. CROP TUMOR ROI
# ============================================================

def crop_tumor_roi(roi):
    """
    Crop the saved tumor ROI to the bounding box
    containing non-zero tumor pixels.
    """

    # Find coordinates of non-zero pixels
    y_coords, x_coords = np.where(
        roi > 0
    )

    # --------------------------------------------------------
    # If no tumor pixels exist
    # --------------------------------------------------------

    if len(x_coords) == 0:

        return None

    # --------------------------------------------------------
    # Find bounding box
    # --------------------------------------------------------

    x_min = np.min(x_coords)
    x_max = np.max(x_coords)

    y_min = np.min(y_coords)
    y_max = np.max(y_coords)

    # --------------------------------------------------------
    # Crop the tumor region
    # +1 because slicing excludes the final index
    # --------------------------------------------------------

    cropped_roi = roi[
        y_min:y_max + 1,
        x_min:x_max + 1
    ]

    return cropped_roi


# ============================================================
# 2. CONVERT ROI TO GLCM GRAY LEVELS
# ============================================================

def prepare_for_glcm(roi):
    """
    Convert ROI intensities into 32 gray levels
    for GLCM computation.
    """

    # Convert to uint8
    roi = roi.astype(
        np.uint8
    )

    # --------------------------------------------------------
    # Handle empty ROI
    # --------------------------------------------------------

    if np.max(roi) == 0:

        return None

    # --------------------------------------------------------
    # Normalize tumor intensities to 0-31
    # --------------------------------------------------------

    roi_normalized = cv2.normalize(
        roi,
        None,
        0,
        GLCM_LEVELS - 1,
        cv2.NORM_MINMAX
    )

    roi_quantized = (
        roi_normalized.astype(np.uint8)
    )

    return roi_quantized


# ============================================================
# 3. EXTRACT GLCM FEATURES
# ============================================================

def extract_glcm_features(roi):
    """
    Extract GLCM texture features from the cropped tumor ROI.

    Features:
        Contrast
        Dissimilarity
        Homogeneity
        Energy
        Correlation
        ASM

    Calculated across:
        4 distances
        4 angles
    """

    # --------------------------------------------------------
    # Crop tumor region
    # --------------------------------------------------------

    cropped_roi = crop_tumor_roi(
        roi
    )

    # --------------------------------------------------------
    # Handle empty ROI
    # --------------------------------------------------------

    if cropped_roi is None:

        return None

    # --------------------------------------------------------
    # Quantize ROI to 32 gray levels
    # --------------------------------------------------------

    roi_quantized = prepare_for_glcm(
        cropped_roi
    )

    if roi_quantized is None:

        return None

    # --------------------------------------------------------
    # Create GLCM
    # --------------------------------------------------------

    glcm = graycomatrix(
        roi_quantized,
        distances=GLCM_DISTANCES,
        angles=GLCM_ANGLES,
        levels=GLCM_LEVELS,
        symmetric=True,
        normed=True
    )

    # --------------------------------------------------------
    # Store extracted features
    # --------------------------------------------------------

    features = []

    # --------------------------------------------------------
    # Extract every property for every distance and angle
    # --------------------------------------------------------

    for property_name in GLCM_PROPERTIES:

        property_values = graycoprops(
            glcm,
            property_name
        )

        for distance_index, distance in enumerate(
            GLCM_DISTANCES
        ):

            for angle_index, angle in enumerate(
                GLCM_ANGLES
            ):

                value = property_values[
                    distance_index,
                    angle_index
                ]

                features.append(
                    value
                )

    return np.array(
        features,
        dtype=np.float32
    )


# ============================================================
# 4. CREATE FEATURE COLUMN NAMES
# ============================================================

def create_feature_names():

    feature_names = []

    for property_name in GLCM_PROPERTIES:

        for distance in GLCM_DISTANCES:

            for angle_index, angle in enumerate(
                GLCM_ANGLES
            ):

                angle_name = [
                    "0",
                    "45",
                    "90",
                    "135"
                ][angle_index]

                feature_name = (
                    f"{property_name}_"
                    f"d{distance}_"
                    f"a{angle_name}"
                )

                feature_names.append(
                    feature_name
                )

    return feature_names


# ============================================================
# 5. CREATE FEATURE NAMES
# ============================================================

FEATURE_NAMES = create_feature_names()


# ============================================================
# 6. DISPLAY INFORMATION
# ============================================================

print("=" * 60)
print("GLCM FUNCTIONS READY")
print("=" * 60)

print("\nNumber of GLCM features:", len(FEATURE_NAMES))

print("\nFeature names:")

for name in FEATURE_NAMES:

    print(" -", name)

GLCM FUNCTIONS READY

Number of GLCM features: 96

Feature names:
 - contrast_d1_a0
 - contrast_d1_a45
 - contrast_d1_a90
 - contrast_d1_a135
 - contrast_d2_a0
 - contrast_d2_a45
 - contrast_d2_a90
 - contrast_d2_a135
 - contrast_d3_a0
 - contrast_d3_a45
 - contrast_d3_a90
 - contrast_d3_a135
 - contrast_d4_a0
 - contrast_d4_a45
 - contrast_d4_a90
 - contrast_d4_a135
 - dissimilarity_d1_a0
 - dissimilarity_d1_a45
 - dissimilarity_d1_a90
 - dissimilarity_d1_a135
 - dissimilarity_d2_a0
 - dissimilarity_d2_a45
 - dissimilarity_d2_a90
 - dissimilarity_d2_a135
 - dissimilarity_d3_a0
 - dissimilarity_d3_a45
 - dissimilarity_d3_a90
 - dissimilarity_d3_a135
 - dissimilarity_d4_a0
 - dissimilarity_d4_a45
 - dissimilarity_d4_a90
 - dissimilarity_d4_a135
 - homogeneity_d1_a0
 - homogeneity_d1_a45
 - homogeneity_d1_a90
 - homogeneity_d1_a135
 - homogeneity_d2_a0
 - homogeneity_d2_a45
 - homogeneity_d2_a90
 - homogeneity_d2_a135
 - homogeneity_d3_a0
 - homogeneity_d3_a45
 - homogeneity_d3_a90
 - ho

# Extract GLCM features and save CSV

In [3]:
# ============================================================
# CELL 3: EXTRACT GLCM FEATURES FROM SAVED ROI DATASET
# ============================================================

import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm


# ============================================================
# FUNCTION: PROCESS ONE DATASET SPLIT
# ============================================================

def extract_features_from_split(
    split_path,
    split_name
):
    """
    Extract GLCM features from all saved ROI images
    in a train or test split.
    """

    all_features = []

    # --------------------------------------------------------
    # Get class folders
    # --------------------------------------------------------

    class_folders = sorted([
        folder
        for folder in split_path.iterdir()
        if folder.is_dir()
    ])

    # --------------------------------------------------------
    # Process each class
    # --------------------------------------------------------

    for class_folder in class_folders:

        class_name = class_folder.name

        # Get ROI image files
        image_files = [
            file
            for file in class_folder.iterdir()
            if file.is_file()
            and file.suffix.lower() in [
                ".jpg",
                ".jpeg",
                ".png",
                ".bmp",
                ".tif",
                ".tiff"
            ]
        ]

        # ----------------------------------------------------
        # tqdm progress bar
        # ----------------------------------------------------

        progress_bar = tqdm(
            image_files,
            desc=f"{split_name} - {class_name}",
            unit="image"
        )

        # ----------------------------------------------------
        # Process every ROI
        # ----------------------------------------------------

        for image_path in progress_bar:

            try:

                # ------------------------------------------------
                # Read saved ROI
                # ------------------------------------------------

                roi = cv2.imread(
                    str(image_path),
                    cv2.IMREAD_GRAYSCALE
                )

                if roi is None:

                    continue

                # ------------------------------------------------
                # Extract GLCM features
                # ------------------------------------------------

                features = extract_glcm_features(
                    roi
                )

                # ------------------------------------------------
                # Handle empty ROI
                # ------------------------------------------------

                if features is None:

                    continue

                # ------------------------------------------------
                # Create one record
                # ------------------------------------------------

                record = {}

                # Add all GLCM features
                for index, feature_name in enumerate(
                    FEATURE_NAMES
                ):

                    record[
                        feature_name
                    ] = features[index]

                # Add class label
                record["class"] = class_name

                # Add original filename
                record["filename"] = image_path.name

                # Store record
                all_features.append(
                    record
                )

                # Update progress bar
                progress_bar.set_postfix(
                    extracted=len(all_features)
                )

            except Exception as e:

                print(
                    f"\nError processing "
                    f"{image_path.name}: {e}"
                )

    # --------------------------------------------------------
    # Convert to DataFrame
    # --------------------------------------------------------

    dataframe = pd.DataFrame(
        all_features
    )

    return dataframe


# ============================================================
# EXTRACT TRAIN FEATURES
# ============================================================

print("\n" + "=" * 60)
print("EXTRACTING GLCM FEATURES - TRAIN")
print("=" * 60)

glcm_train_df = extract_features_from_split(
    ROI_TRAIN_PATH,
    "TRAIN"
)


# ============================================================
# EXTRACT TEST FEATURES
# ============================================================

print("\n" + "=" * 60)
print("EXTRACTING GLCM FEATURES - TEST")
print("=" * 60)

glcm_test_df = extract_features_from_split(
    ROI_TEST_PATH,
    "TEST"
)


# ============================================================
# SAVE TRAIN CSV
# ============================================================

TRAIN_FEATURE_FILE = (
    FEATURE_PATH / "glcm_train.csv"
)

glcm_train_df.to_csv(
    TRAIN_FEATURE_FILE,
    index=False
)


# ============================================================
# SAVE TEST CSV
# ============================================================

TEST_FEATURE_FILE = (
    FEATURE_PATH / "glcm_test.csv"
)

glcm_test_df.to_csv(
    TEST_FEATURE_FILE,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("GLCM FEATURE EXTRACTION COMPLETE")
print("=" * 60)

print(
    "\nTraining samples:",
    len(glcm_train_df)
)

print(
    "Testing samples:",
    len(glcm_test_df)
)

print(
    "\nNumber of GLCM features:",
    len(FEATURE_NAMES)
)

print(
    "\nTraining CSV:",
    TRAIN_FEATURE_FILE
)

print(
    "Testing CSV:",
    TEST_FEATURE_FILE
)


EXTRACTING GLCM FEATURES - TRAIN


TRAIN - glioma:   0%|          | 0/1108 [00:00<?, ?image/s]

TRAIN - meningioma:   0%|          | 0/1320 [00:00<?, ?image/s]

TRAIN - pituitary:   0%|          | 0/1455 [00:00<?, ?image/s]


EXTRACTING GLCM FEATURES - TEST


TEST - glioma:   0%|          | 0/234 [00:00<?, ?image/s]

TEST - meningioma:   0%|          | 0/301 [00:00<?, ?image/s]

TEST - pituitary:   0%|          | 0/295 [00:00<?, ?image/s]


GLCM FEATURE EXTRACTION COMPLETE

Training samples: 3883
Testing samples: 830

Number of GLCM features: 96

Training CSV: C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)\glcm_train.csv
Testing CSV: C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)\glcm_test.csv


# Standardization + PCA with 95% variance

In [9]:
# ============================================================
# CELL 4: STANDARDIZATION + PCA
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


# ============================================================
# 1. SEPARATE FEATURES AND LABELS
# ============================================================

# GLCM feature columns
X_train = glcm_train_df[
    FEATURE_NAMES
].values

X_test = glcm_test_df[
    FEATURE_NAMES
].values


# Class labels
y_train = glcm_train_df[
    "class"
].values

y_test = glcm_test_df[
    "class"
].values


# ============================================================
# 2. STANDARDIZE FEATURES
# ============================================================
#
# GLCM features have different numerical ranges.
# Standardization puts them on a comparable scale.
#
# IMPORTANT:
# Fit ONLY on training data.
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


# ============================================================
# 3. PCA
# ============================================================
#
# Instead of manually selecting 6 components,
# retain 95% of the variance.
#
# PCA automatically determines the required
# number of components.
# ============================================================

pca = PCA(
    n_components=50,
    random_state=42
)


# Fit PCA ONLY on training data
X_train_pca = pca.fit_transform(
    X_train_scaled
)


# Apply the SAME PCA transformation to test data
X_test_pca = pca.transform(
    X_test_scaled
)


# ============================================================
# 4. PCA INFORMATION
# ============================================================

print("=" * 60)
print("PCA RESULTS")
print("=" * 60)

print(
    "\nOriginal number of features:",
    X_train.shape[1]
)

print(
    "Number of PCA components:",
    X_train_pca.shape[1]
)

print(
    "Training shape after PCA:",
    X_train_pca.shape
)

print(
    "Testing shape after PCA:",
    X_test_pca.shape
)

print(
    "\nTotal explained variance:",
    round(
        np.sum(
            pca.explained_variance_ratio_
        ) * 100,
        2
    ),
    "%"
)


# ============================================================
# 5. SHOW VARIANCE CONTRIBUTION
# ============================================================

print("\nVariance explained by each component:")

for i, variance in enumerate(
    pca.explained_variance_ratio_,
    start=1
):

    print(
        f"PC{i}: {variance * 100:.2f}%"
    )

PCA RESULTS

Original number of features: 96
Number of PCA components: 50
Training shape after PCA: (3883, 50)
Testing shape after PCA: (830, 50)

Total explained variance: 99.99 %

Variance explained by each component:
PC1: 54.90%
PC2: 24.28%
PC3: 7.91%
PC4: 4.77%
PC5: 3.78%
PC6: 1.95%
PC7: 0.75%
PC8: 0.44%
PC9: 0.21%
PC10: 0.17%
PC11: 0.14%
PC12: 0.11%
PC13: 0.10%
PC14: 0.08%
PC15: 0.06%
PC16: 0.05%
PC17: 0.04%
PC18: 0.03%
PC19: 0.03%
PC20: 0.02%
PC21: 0.02%
PC22: 0.02%
PC23: 0.02%
PC24: 0.01%
PC25: 0.01%
PC26: 0.01%
PC27: 0.01%
PC28: 0.01%
PC29: 0.01%
PC30: 0.01%
PC31: 0.00%
PC32: 0.00%
PC33: 0.00%
PC34: 0.00%
PC35: 0.00%
PC36: 0.00%
PC37: 0.00%
PC38: 0.00%
PC39: 0.00%
PC40: 0.00%
PC41: 0.00%
PC42: 0.00%
PC43: 0.00%
PC44: 0.00%
PC45: 0.00%
PC46: 0.00%
PC47: 0.00%
PC48: 0.00%
PC49: 0.00%
PC50: 0.00%


# Save PCA Dataset:

In [10]:
# ============================================================
# CELL 5: SAVE PCA FEATURES FOR SVM AND KNN
# ============================================================

import pandas as pd


# ============================================================
# 1. CREATE PCA COLUMN NAMES
# ============================================================

pca_feature_names = [
    f"PC{i}"
    for i in range(
        1,
        X_train_pca.shape[1] + 1
    )
]


# ============================================================
# 2. CREATE TRAINING PCA DATAFRAME
# ============================================================

pca_train_df = pd.DataFrame(
    X_train_pca,
    columns=pca_feature_names
)

# Add class label
pca_train_df["class"] = y_train

# Add filename
pca_train_df["filename"] = (
    glcm_train_df["filename"].values
)


# ============================================================
# 3. CREATE TESTING PCA DATAFRAME
# ============================================================

pca_test_df = pd.DataFrame(
    X_test_pca,
    columns=pca_feature_names
)

# Add class label
pca_test_df["class"] = y_test

# Add filename
pca_test_df["filename"] = (
    glcm_test_df["filename"].values
)


# ============================================================
# 4. SAVE TRAIN PCA CSV
# ============================================================

PCA_TRAIN_FILE = (
    FEATURE_PATH / "pca_train.csv"
)

pca_train_df.to_csv(
    PCA_TRAIN_FILE,
    index=False
)


# ============================================================
# 5. SAVE TEST PCA CSV
# ============================================================

PCA_TEST_FILE = (
    FEATURE_PATH / "pca_test.csv"
)

pca_test_df.to_csv(
    PCA_TEST_FILE,
    index=False
)


# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("=" * 60)
print("PCA DATASET SAVED")
print("=" * 60)

print(
    "\nTraining samples:",
    len(pca_train_df)
)

print(
    "Testing samples:",
    len(pca_test_df)
)

print(
    "PCA components:",
    len(pca_feature_names)
)

print(
    "\nTraining CSV:"
)

print(PCA_TRAIN_FILE)

print(
    "\nTesting CSV:"
)

print(PCA_TEST_FILE)


# ============================================================
# 7. SHOW FIRST FEW ROWS
# ============================================================

print("\nFirst 5 training samples:")

display(
    pca_train_df.head()
)

PCA DATASET SAVED

Training samples: 3883
Testing samples: 830
PCA components: 50

Training CSV:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)\pca_train.csv

Testing CSV:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)\pca_test.csv

First 5 training samples:


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC43,PC44,PC45,PC46,PC47,PC48,PC49,PC50,class,filename
0,7.263428,-6.013046,-4.168631,-0.556080,-0.127549,-0.079840,-0.397175,-0.249179,-0.175544,0.057873,...,0.015196,0.005576,0.017753,-0.000738,0.000892,-0.021387,-0.008519,-0.011329,glioma,brisc2025_train_00001_gl_ax_t1.jpg
1,-1.006626,-6.170053,-3.909963,0.954230,-0.612781,0.493682,0.872141,0.162144,-0.116123,0.075686,...,-0.041050,0.010738,-0.015824,-0.026861,-0.000424,0.033067,0.006125,-0.020362,glioma,brisc2025_train_00002_gl_ax_t1.jpg
2,-17.139891,-1.423025,1.203283,-4.316956,-2.530144,0.163525,0.968133,1.796074,-0.452670,0.563299,...,-0.021746,-0.067159,-0.087027,-0.020925,0.074998,-0.059692,-0.021860,-0.023023,glioma,brisc2025_train_00003_gl_ax_t1.jpg
3,6.493587,1.783234,3.275978,-1.545292,0.250585,0.089056,0.011509,-0.485299,-0.581549,-0.133385,...,0.003796,-0.007134,0.002206,-0.007906,0.000425,-0.004953,-0.015480,0.008030,glioma,brisc2025_train_00004_gl_ax_t1.jpg
4,4.401727,0.500472,-0.056520,-0.993232,-0.674783,0.869633,0.553542,-0.844351,0.014723,0.041904,...,0.060127,-0.022438,-0.017846,-0.020188,0.029560,-0.004515,0.028862,0.004903,glioma,brisc2025_train_00005_gl_ax_t1.jpg


# SAVE TRAINED SCALER AND PCA

In [11]:
# ============================================================
# CELL 6: SAVE TRAINED SCALER AND PCA
# ============================================================

import joblib


# ============================================================
# 1. SAVE STANDARD SCALER
# ============================================================

SCALER_FILE = (
    FEATURE_PATH / "scaler.pkl"
)

joblib.dump(
    scaler,
    SCALER_FILE
)


# ============================================================
# 2. SAVE PCA MODEL
# ============================================================

PCA_FILE = (
    FEATURE_PATH / "pca.pkl"
)

joblib.dump(
    pca,
    PCA_FILE
)


# ============================================================
# 3. DISPLAY SAVED FILES
# ============================================================

print("=" * 60)
print("SCALER AND PCA SAVED")
print("=" * 60)

print("\nScaler saved at:")
print(SCALER_FILE)

print("\nPCA saved at:")
print(PCA_FILE)

print("\nPCA components:", pca.n_components_)

print(
    "Explained variance:",
    round(
        np.sum(
            pca.explained_variance_ratio_
        ) * 100,
        2
    ),
    "%"
)

SCALER AND PCA SAVED

Scaler saved at:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)\scaler.pkl

PCA saved at:
C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Features(1)\pca.pkl

PCA components: 50
Explained variance: 99.99 %


# Find missing test ROIs

In [7]:
# ============================================================
# CELL 6: CHECK MISSING TEST SAMPLES
# ============================================================

from pathlib import Path


# ------------------------------------------------------------
# ORIGINAL PREPROCESSED CLASSIFICATION TEST DATA
# ------------------------------------------------------------

original_test_path = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Classification_Preprocessed\test"
)


# ------------------------------------------------------------
# Get class names
# ------------------------------------------------------------

test_class_names = sorted([
    folder.name
    for folder in original_test_path.iterdir()
    if folder.is_dir()
])


# ------------------------------------------------------------
# Get all original test images
# ------------------------------------------------------------

original_test_files = []

for class_name in test_class_names:

    class_folder = (
        original_test_path / class_name
    )

    image_files = [
        file
        for file in class_folder.iterdir()
        if file.is_file()
        and file.suffix.lower() in [
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp",
            ".tif",
            ".tiff"
        ]
    ]

    for image_path in image_files:

        original_test_files.append(
            (
                class_name,
                image_path.name
            )
        )


# ------------------------------------------------------------
# Get successfully extracted GLCM samples
# ------------------------------------------------------------

extracted_test_files = set(
    zip(
        glcm_test_df["class"],
        glcm_test_df["filename"]
    )
)


# ------------------------------------------------------------
# Find missing samples
# ------------------------------------------------------------

missing_test_files = []

for class_name, filename in original_test_files:

    if (
        class_name,
        filename
    ) not in extracted_test_files:

        missing_test_files.append(
            (
                class_name,
                filename
            )
        )


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 60)
print("MISSING TEST SAMPLE CHECK")
print("=" * 60)

print(
    "\nOriginal test images:",
    len(original_test_files)
)

print(
    "GLCM extracted samples:",
    len(extracted_test_files)
)

print(
    "Missing samples:",
    len(missing_test_files)
)


# ------------------------------------------------------------
# Display missing files
# ------------------------------------------------------------

if len(missing_test_files) > 0:

    print("\nMissing test images:")
    print("-" * 60)

    for class_name, filename in missing_test_files:

        print(
            f"{class_name:15} {filename}"
        )

else:

    print("\nNo missing test samples found.")

MISSING TEST SAMPLE CHECK

Original test images: 860
GLCM extracted samples: 830
Missing samples: 30

Missing test images:
------------------------------------------------------------
glioma          brisc2025_test_00004_gl_ax_t1.jpg
glioma          brisc2025_test_00005_gl_ax_t1.jpg
glioma          brisc2025_test_00007_gl_ax_t1.jpg
glioma          brisc2025_test_00009_gl_ax_t1.jpg
glioma          brisc2025_test_00010_gl_ax_t1.jpg
glioma          brisc2025_test_00025_gl_ax_t1.jpg
glioma          brisc2025_test_00045_gl_ax_t1.jpg
glioma          brisc2025_test_00056_gl_ax_t1.jpg
glioma          brisc2025_test_00057_gl_ax_t1.jpg
glioma          brisc2025_test_00059_gl_ax_t1.jpg
glioma          brisc2025_test_00060_gl_ax_t1.jpg
glioma          brisc2025_test_00069_gl_ax_t1.jpg
glioma          brisc2025_test_00085_gl_ax_t1.jpg
glioma          brisc2025_test_00096_gl_co_t1.jpg
glioma          brisc2025_test_00131_gl_co_t1.jpg
glioma          brisc2025_test_00140_gl_co_t1.jpg
glioma          

# Check the 30 missing ROI images

In [8]:
# ============================================================
# CELL 7: INSPECT MISSING TEST ROI IMAGES
# ============================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ------------------------------------------------------------
# ROI TEST DATASET
# ------------------------------------------------------------

roi_test_path = Path(
    r"C:\Users\harsh\OneDrive\Documents\BE_Major_Project\DATASET\Tumor_ROI\test"
)


# ------------------------------------------------------------
# Store information about missing ROIs
# ------------------------------------------------------------

missing_roi_info = []


# ------------------------------------------------------------
# Check every missing image
# ------------------------------------------------------------

for class_name, filename in missing_test_files:

    roi_path = (
        roi_test_path
        / class_name
        / filename
    )

    # Check file existence
    if not roi_path.exists():

        missing_roi_info.append(
            (
                class_name,
                filename,
                "FILE NOT FOUND",
                0
            )
        )

        continue

    # Read ROI
    roi = cv2.imread(
        str(roi_path),
        cv2.IMREAD_GRAYSCALE
    )

    if roi is None:

        missing_roi_info.append(
            (
                class_name,
                filename,
                "READ ERROR",
                0
            )
        )

        continue

    # Count non-zero pixels
    nonzero_pixels = np.count_nonzero(
        roi
    )

    # Determine status
    if nonzero_pixels == 0:

        status = "EMPTY ROI"

    else:

        status = "HAS TUMOR PIXELS"

    missing_roi_info.append(
        (
            class_name,
            filename,
            status,
            nonzero_pixels
        )
    )


# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

missing_roi_df = pd.DataFrame(
    missing_roi_info,
    columns=[
        "class",
        "filename",
        "status",
        "nonzero_pixels"
    ]
)


# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("=" * 60)
print("MISSING ROI ANALYSIS")
print("=" * 60)

print(
    "\nTotal missing samples:",
    len(missing_roi_df)
)

print(
    "\nStatus distribution:"
)

print(
    missing_roi_df["status"].value_counts()
)


# ------------------------------------------------------------
# Display full table
# ------------------------------------------------------------

display(
    missing_roi_df
)


# ------------------------------------------------------------
# Display non-empty ROIs if any
# ------------------------------------------------------------

non_empty = missing_roi_df[
    missing_roi_df["status"] == "HAS TUMOR PIXELS"
]


if len(non_empty) > 0:

    print("\nNon-empty missing ROIs found.")

    display(
        non_empty
    )

else:

    print(
        "\nAll missing ROIs are empty."
    )

MISSING ROI ANALYSIS

Total missing samples: 30

Status distribution:
status
EMPTY ROI    30
Name: count, dtype: int64


,class,filename,status,nonzero_pixels
0,glioma,brisc2025_test_00004_gl_ax_t1.jpg,EMPTY ROI,0
1,glioma,brisc2025_test_00005_gl_ax_t1.jpg,EMPTY ROI,0
2,glioma,brisc2025_test_00007_gl_ax_t1.jpg,EMPTY ROI,0
3,glioma,brisc2025_test_00009_gl_ax_t1.jpg,EMPTY ROI,0
4,glioma,brisc2025_test_00010_gl_ax_t1.jpg,EMPTY ROI,0
5,glioma,brisc2025_test_00025_gl_ax_t1.jpg,EMPTY ROI,0
6,glioma,brisc2025_test_00045_gl_ax_t1.jpg,EMPTY ROI,0
7,glioma,brisc2025_test_00056_gl_ax_t1.jpg,EMPTY ROI,0
8,glioma,brisc2025_test_00057_gl_ax_t1.jpg,EMPTY ROI,0
9,glioma,brisc2025_test_00059_gl_ax_t1.jpg,EMPTY ROI,0



All missing ROIs are empty.
